# 01 Motor Speed Tuning

This notebook launches the dedicated motor calibration panel:

- raw `L / R / T` motor tests
- save `forward`
- save `rotate_left`
- save `rotate_right`
- save `forward_nudge`

Saved presets go to `.jetcar_motor_calibration.json` in the project root.


## Safety

Put the rover on a stand first. Keep one hand ready near stop. Do not move to floor driving until the saved presets feel stable.


In [ ]:
import glob
import os
import socket
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
serial_candidates = [p for p in ('/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]
hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})

print('Project root:', project_root)
print('Serial candidates:', serial_candidates or ['none found'])
print('Calibration file:', project_root / '.jetcar_motor_calibration.json')
print('Host IPv4:', ip_addresses or ['127.0.0.1'])


In [ ]:
host = '0.0.0.0'
http_port = 8766
serial_port = serial_candidates[0] if serial_candidates else '/dev/ttyTHS1'
baudrate = 115200

print('Edit this cell if your serial port or port number is different, then run the next cells.')


In [ ]:
import shlex
import subprocess
import sys
import time
from IPython.display import HTML, display

if '_motor_calibration_process' not in globals():
    _motor_calibration_process = None

panel_log_path = project_root / '.jetcar_motor_calibration_panel.notebook.log'

def panel_python() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable


def panel_command() -> list[str]:
    return [
        panel_python(),
        str(project_root / 'scripts' / 'jetcar_motor_calibration_panel.py'),
        '--host', host,
        '--http-port', str(http_port),
        '--port', serial_port,
        '--baudrate', str(baudrate),
    ]


def stop_motor_calibration_panel() -> None:
    global _motor_calibration_process
    if _motor_calibration_process is None:
        return
    if _motor_calibration_process.poll() is None:
        _motor_calibration_process.terminate()
        try:
            _motor_calibration_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _motor_calibration_process.kill()
            _motor_calibration_process.wait(timeout=3)
    _motor_calibration_process = None


def browser_url() -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{http_port}'

print('Equivalent terminal command:')
print(shlex.join(panel_command()))


In [ ]:
stop_motor_calibration_panel()
cmd = panel_command()
print('Starting motor calibration panel...')
print(shlex.join(cmd))
with open(panel_log_path, 'w') as log_file:
    _motor_calibration_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
time.sleep(2.0)
if _motor_calibration_process.poll() is not None:
    raise RuntimeError(f'motor calibration panel exited early; check {panel_log_path}')
print('Motor calibration panel running at:', browser_url())
if ip_addresses:
    print('LAN example:', f'http://{ip_addresses[0]}:{http_port}')
display(HTML(
    f'<p><a href="{browser_url()}" target="_blank">Open the motor calibration panel in a new tab</a></p>'
    f'<iframe src="{browser_url()}" width="100%" height="780" style="border:1px solid #ccc; border-radius:12px;"></iframe>'
))


In [ ]:
print('Last motor calibration panel log lines:')
if panel_log_path.exists():
    print('\n'.join(panel_log_path.read_text().splitlines()[-40:]))
else:
    print('No log file yet.')


In [ ]:
# stop_motor_calibration_panel()
# print('Motor calibration panel stopped.')


## What To Do In The Panel

1. make sure `Serial` becomes ready
2. use `Run Raw Test` for small test pulses first
3. save a reliable `forward` preset
4. save `rotate_left` and `rotate_right` so the rover turns cleanly
5. save `forward_nudge` for the short post-turn refresh
6. confirm the JSON preview looks right
7. then move to `02_line_following.ipynb`

The line-following notebooks reuse these presets automatically.
